# SMAP Soil Moisture Drought Trigger for U.S. Counties

**Author:** Frank Anyoka Adekilae, Louisiana State University

Code accompanying the manuscript *"Soil Moisture Active Passive (SMAP) Surface Soil Moisture as an Early-Warning Indicator of Subsequent County-Scale Drought Conditions in the Conterminous United States: A Benchmark Against the U.S. Drought Monitor, 2016–2020"* (submitted to the *Journal of Applied Remote Sensing*).

The default settings reproduce the paper (Eqs. 1–8 and Table 1). Optional sensitivity analyses that are **not** in the paper are clearly marked in Section 10.

**Workflow**
1. Clean daily SMAP 9 km soil moisture (one CSV per year exported from Google Earth Engine) and save as Parquet.
2. Assign each SMAP pixel to the county containing its center.
3. Average soil moisture to county-daily values (Eq. 1).
4. Standardize against each county's 2016–2020 mean and standard deviation (Eqs. 2–5); percentile anomalies for comparison.
5. Dryness trigger: z ≤ −1.0 on at least 3 of the calendar days t−4 … t (Eq. 6).
6. Future event labels: DSCI ≥ 200 on any USDM map date in (t, t+L] days (Eq. 7).
7. Verification of the binary trigger: ROC–AUC, PR–AUC, Brier score (Eq. 8), Brier skill score, POD, POFD (Table 1).

**Note on the event definition.** DSCI ≥ 200 is a study-specific level of combined drought severity and coverage. DSCI does not map uniquely to one USDM category, so it should not be read as "county-wide D1 or worse".

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

In [ ]:
# ---- Configuration: edit these paths for your setup -------------------------
DATA_DIR = "data"          # folder with SMAP_9km_pixels_<year>.csv and the USDM CSV
OUT_DIR  = "outputs"       # all processed files and figures are written here
YEARS    = range(2016, 2021)

SMAP_CSV_PATTERN = os.path.join(DATA_DIR, "SMAP_9km_pixels_{year}.csv")
USDM_CSV         = os.path.join(DATA_DIR, "USDM_county_2016_2020.csv")
COUNTIES_SOURCE  = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_5m.zip"

# Method settings (paper values)
SM_MIN, SM_MAX   = 0.02, 0.70  # physically plausible volumetric soil moisture (m3/m3), Sec. 2.2
Z_THRESH         = -1.0        # a day is "dry" when z <= this value (Eq. 6)
TRIGGER_WINDOW   = "5D"        # calendar days t-4 ... t (Eq. 6)
TRIGGER_MIN_DAYS = 3           # dry days needed in the window
DSCI_EVENT       = 200         # event = DSCI >= 200 (Sec. 2.4)
LEAD_DAYS        = [7, 14, 21, 30, 45, 60]

CONUS_EXCLUDE = {"02", "15", "60", "66", "69", "72", "78"}   # AK, HI, and territories
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Optional: run in Google Colab with data on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/SMAP_data"
# OUT_DIR  = "/content/drive/MyDrive/SMAP_outputs"
# (then re-run the configuration cell's path lines and os.makedirs)

## 2. Clean yearly SMAP files

Each CSV (exported from Google Earth Engine, descending-overpass `soil_moisture_am` masked to `retrieval_qual_flag_am = 0`) has one row per pixel per day with `date`, `lat`, `lon`, and `sm`. Values outside 0.02–0.70 m³ m⁻³ are removed, coordinates are rounded to form a stable pixel key, and compact Parquet files are saved.

In [ ]:
def preprocess_smap_year(year):
    df = pd.read_csv(SMAP_CSV_PATTERN.format(year=year),
                     usecols=lambda c: c in {"date", "lat", "lon", "sm"})
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df = df.dropna(subset=["sm"])
    df = df[df["sm"].between(SM_MIN, SM_MAX)]
    # Rounded coordinates act as a stable pixel key (safer than joining on raw floats)
    df["lat"] = df["lat"].round(4).astype("float32")
    df["lon"] = df["lon"].round(4).astype("float32")
    df["sm"] = df["sm"].astype("float32")
    out = os.path.join(OUT_DIR, f"smap_{year}.parquet")
    df.to_parquet(out, index=False)
    print(f"{year}: {len(df):,} rows -> {out}")

for year in YEARS:
    preprocess_smap_year(year)

## 3. Assign SMAP pixels to counties

Each pixel is assigned to the 2020 Census county that contains the pixel center (Sec. 2.3). Counties smaller than a 9 km pixel may receive no pixel.

In [ ]:
counties = gpd.read_file(COUNTIES_SOURCE)[["GEOID", "STATEFP", "geometry"]]
counties = counties[~counties["STATEFP"].isin(CONUS_EXCLUDE)].to_crs("EPSG:4326")

pixels = pd.concat(
    [pd.read_parquet(os.path.join(OUT_DIR, f"smap_{y}.parquet"), columns=["lat", "lon"]).drop_duplicates()
     for y in YEARS]
).drop_duplicates()

pts = gpd.GeoDataFrame(pixels, geometry=gpd.points_from_xy(pixels["lon"], pixels["lat"]), crs="EPSG:4326")
joined = gpd.sjoin(pts, counties[["GEOID", "geometry"]], how="inner", predicate="within")
pixel_to_county = (joined[["lat", "lon", "GEOID"]]
                   .rename(columns={"GEOID": "county_fips"})
                   .drop_duplicates(subset=["lat", "lon"]))
pixel_to_county.to_parquet(os.path.join(OUT_DIR, "pixel_to_county.parquet"), index=False)
print(f"{len(pixel_to_county):,} pixels mapped to {pixel_to_county['county_fips'].nunique():,} counties")

## 4. County-daily soil moisture (Eq. 1)

In [ ]:
parts = []
for y in YEARS:
    df = pd.read_parquet(os.path.join(OUT_DIR, f"smap_{y}.parquet"), columns=["lat", "lon", "date", "sm"])
    df = df.merge(pixel_to_county, on=["lat", "lon"], how="inner")
    parts.append(df.groupby(["county_fips", "date"], as_index=False)["sm"].mean()
                   .rename(columns={"sm": "sm_mean"}))
county = pd.concat(parts, ignore_index=True).sort_values(["county_fips", "date"]).reset_index(drop=True)
county.to_parquet(os.path.join(OUT_DIR, "county_daily_sm.parquet"), index=False)
print(county.shape, "| counties:", county["county_fips"].nunique())
county.head()

## 5. Standardized and percentile anomalies (Eqs. 2–5)

For each county, the anomaly is taken relative to that county's 2016–2020 mean (Eq. 2) and sample standard deviation (Eq. 3) over all available days: `sm_anom = sm_mean − μ` (Eq. 4) and `sm_z = sm_anom / σ` (Eq. 5). The percentile anomaly ranks each county-day value within the county's full record, rescaled to [0, 1] (Sec. 3.3).

In [ ]:
def add_county_zscore(df):
    df = df.copy()
    g = df.groupby("county_fips")["sm_mean"]
    df["mu"] = g.transform("mean")                 # Eq. 2
    df["sigma"] = g.transform("std")               # Eq. 3 (ddof = 1)
    df["sm_anom"] = df["sm_mean"] - df["mu"]       # Eq. 4
    df["sm_z"] = df["sm_anom"] / df["sigma"]       # Eq. 5
    df["sm_pct"] = g.rank(pct=True)                # percentile anomaly in (0, 1]
    return df

county = add_county_zscore(county)
county[["sm_mean", "sm_anom", "sm_z", "sm_pct"]].describe()

## 6. Dryness trigger (Eq. 6)

A county triggers on day t when z ≤ −1.0 on at least 3 of the **calendar** days t−4 … t. A time-based window is used because SMAP 9 km has missing days.

In [ ]:
def add_trigger(df):
    df = df.sort_values(["county_fips", "date"]).reset_index(drop=True)
    df["dry"] = (df["sm_z"] <= Z_THRESH).astype(int)
    counts = []
    for _, g in df.groupby("county_fips", sort=True):
        counts.append(g.set_index("date")["dry"].rolling(TRIGGER_WINDOW).sum().values)
    df["dry_count"] = np.concatenate(counts)
    df["trigger"] = (df["dry_count"] >= TRIGGER_MIN_DAYS).astype(int)
    return df

county = add_trigger(county)
county.to_parquet(os.path.join(OUT_DIR, "county_daily_sm_anom_trigger.parquet"), index=False)
print("Share of county-days with trigger on:", round(county["trigger"].mean(), 3))

## 7. U.S. Drought Monitor DSCI events

USDM county statistics are weekly. An event is `DSCI >= 200` (Sec. 2.4). The full weekly record is kept for building future labels, so a missing SMAP day never hides a future event.

In [ ]:
usdm = pd.read_csv(USDM_CSV)
usdm.columns = usdm.columns.str.strip()
usdm["county_fips"] = usdm["FIPS"].astype(str).str.zfill(5)
usdm["date"] = pd.to_datetime(usdm["MapDate"].astype(str), format="%Y%m%d")
usdm["event"] = (usdm["DSCI"] >= DSCI_EVENT).astype(int)
usdm = (usdm[usdm["county_fips"].isin(county["county_fips"].unique())]
        [["county_fips", "date", "DSCI", "event"]]
        .sort_values(["county_fips", "date"]).reset_index(drop=True))
last_map_date = usdm["date"].max()
print(len(usdm), "county-map rows | last USDM map:", last_map_date.date())

## 8. Verification pairs and future-event labels (Eq. 7)

Verification pairs are county × USDM map dates with valid SMAP on that date (Sec. 2.5). For each lead window L, the label is 1 if DSCI ≥ 200 on any USDM map date in (t, t+L] days, using the county's full USDM record. Pairs whose window extends past the last USDM map are excluded (label left empty). Counties already at or above the threshold at t are **not** excluded, as in the paper.

In [ ]:
pairs = county.merge(usdm, on=["county_fips", "date"], how="inner")
pairs = pairs.sort_values(["county_fips", "date"]).reset_index(drop=True)
print("Verification pairs:", len(pairs), "| counties:", pairs["county_fips"].nunique())

usdm_by_county = {k: g for k, g in usdm.groupby("county_fips", sort=False)}
idx_by_county = pairs.groupby("county_fips", sort=False).indices
for L in LEAD_DAYS:
    lab = np.full(len(pairs), np.nan)
    for fips, idx in idx_by_county.items():
        g = usdm_by_county[fips]
        d = g["date"].values
        cs = np.concatenate([[0], np.cumsum(g["event"].values)])
        t = pairs["date"].values[idx]
        lo = np.searchsorted(d, t, side="right")                            # first map date > t
        hi = np.searchsorted(d, t + np.timedelta64(L, "D"), side="right")   # map dates <= t + L
        vals = ((cs[hi] - cs[lo]) > 0).astype(float)
        vals[(t + np.timedelta64(L, "D")) > np.datetime64(last_map_date)] = np.nan
        lab[idx] = vals
    pairs[f"event_next_{L}d"] = lab
pairs.to_parquet(os.path.join(OUT_DIR, "verification_pairs_labeled.parquet"), index=False)
pairs.head()

## 9. Verification of the binary trigger (Table 1)

The forecast is the binary trigger (0/1). Scores: event rate, ROC–AUC, PR–AUC (average precision), Brier score (Eq. 8), Brier skill score against climatology `BS_ref = p̄(1 − p̄)`, POD (hit rate), and POFD (false-alarm rate).

In [ ]:
def verify(df, score_col="trigger", onset_only=False):
    rows = []
    for L in LEAD_DAYS:
        d = df.dropna(subset=[f"event_next_{L}d", score_col])
        if onset_only:
            d = d[d["event"] == 0]
        y = d[f"event_next_{L}d"].astype(int).values
        f = d[score_col].astype(float).values
        if len(y) == 0 or y.min() == y.max():
            continue
        p = y.mean()
        row = {"lead_days": L, "N": len(y), "event_rate": p,
               "ROC_AUC": roc_auc_score(y, f), "PR_AUC": average_precision_score(y, f)}
        if set(np.unique(f)) <= {0.0, 1.0}:          # binary forecast: Brier and contingency scores
            bs = brier_score_loss(y, f)
            row.update(BS=bs, BSS=1 - bs / (p * (1 - p)),
                       POD=f[y == 1].mean(), POFD=f[y == 0].mean())
        rows.append(row)
    return pd.DataFrame(rows)

table1 = verify(pairs)
table1.round(3).to_csv(os.path.join(OUT_DIR, "Table1.csv"), index=False)
table1.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].plot(table1["lead_days"], table1["ROC_AUC"], marker="o", lw=2, label="ROC–AUC")
axes[0].axhline(0.5, color="grey", lw=1, ls=":", label="No skill (0.5)")
axes[0].set_ylabel("ROC–AUC")
axes[1].plot(table1["lead_days"], table1["PR_AUC"], marker="o", lw=2, label="PR–AUC")
axes[1].plot(table1["lead_days"], table1["event_rate"], marker="s", lw=1.5, ls="--", label="Event-rate baseline")
axes[1].set_ylabel("PR–AUC")
for ax, t in zip(axes, ["(a)", "(b)"]):
    ax.set_xlabel("Lead window (days)"); ax.set_xticks(LEAD_DAYS)
    ax.set_title(t, loc="left"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "skill_vs_lead_window.png"), dpi=300)
plt.show()

## 10. Optional sensitivity analyses (not reported in the paper)

These extensions follow the future-work directions in Sec. 4 of the paper:
- **Continuous score:** ROC–AUC and PR–AUC using −z (drier = higher score) instead of the binary trigger.
- **Onset only:** keep only forecast dates when the county is below DSCI 200, so the test measures development rather than persistence.

In [ ]:
pairs["neg_z"] = -pairs["sm_z"]
sens = {
    "continuous_score_all":   verify(pairs, score_col="neg_z"),
    "binary_trigger_onset":   verify(pairs, onset_only=True),
    "continuous_score_onset": verify(pairs, score_col="neg_z", onset_only=True),
}
for name, t in sens.items():
    t.round(3).to_csv(os.path.join(OUT_DIR, f"sensitivity_{name}.csv"), index=False)
    print(f"\n=== {name} ===")
    print(t.round(3).to_string(index=False))

## 11. Example: one county time series

In [ ]:
def plot_county(fips):
    c = county[county["county_fips"] == fips]
    if c.empty:
        print("No data for", fips)
        return
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    axes[0].plot(c["date"], c["sm_mean"], lw=0.8)
    axes[0].set_ylabel("SM (m³ m⁻³)")
    axes[1].plot(c["date"], c["sm_z"], lw=0.8)
    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].fill_between(c["date"], c["sm_z"].min(), c["sm_z"].max(),
                         where=c["trigger"] == 1, alpha=0.2, label="Trigger on")
    axes[1].set_ylabel("SM z-score")
    axes[1].legend(loc="lower left")
    axes[0].set_title(f"County {fips}: SMAP soil moisture and standardized anomaly")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"county_{fips}_timeseries.png"), dpi=300)
    plt.show()

plot_county(county["county_fips"].iloc[0])   # or a FIPS code, e.g. "22033" (East Baton Rouge)